In [3]:
# 1. Imports

from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.vectorstores import FAISS
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

# 2. Setup LLM and Embeddings
llm = ChatOllama(
    model = 'mistral',
    temperature=0)
embedding = OllamaEmbeddings(
    model='nomic-embed-text'
)

# 3. Load your document
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 4. Split it into chunks
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(raw_text)
documents = [Document(page_content=chunk) for chunk in chunks]

# 5. Compress the documents using LLMChainExtractor
compressor = LLMChainExtractor.from_llm(llm)
compressed_docs = compressor.compress_documents(documents, query="Summarize the key points")

print("✅ Compressed Summary:")
for doc in compressed_docs:
    print("-", doc.page_content)

# 6. Now ask a question about the compressed content using a custom LLMChain
qa_prompt = PromptTemplate.from_template(
    "Given the context below, answer the question:\n\nContext:\n{context}\n\nQuestion: {question}"
)
qa_chain = qa_prompt | llm

response = qa_chain.invoke({
    "context": "\n".join([doc.page_content for doc in compressed_docs]),
    "question": "What is the main idea of the document in bullet points?"
})

print("\n💡 Answer to your question:")
print(response)


Created a chunk of size 622, which is longer than the specified 300
Created a chunk of size 803, which is longer than the specified 300


✅ Compressed Summary:
- Large Language Models (LLMs) have transformed the landscape of artificial intelligence by enabling machines to understand, generate, and reason with human language at an unprecedented scale. Built on the transformer architecture, models like GPT-4, Llama 4, and Gemini process vast datasets to predict the next token in a sequence, allowing them to perform tasks ranging from creative writing to complex code generation. However, despite their "intelligence," LLMs are essentially isolated engines; they lack a built-in memory of past interactions and cannot natively interact with external data sources or software tools.
- This is where **LangChain** becomes essential.
LangChain is an open-source framework designed to bridge the gap between static LLMs and dynamic, data-aware applications.
It allows developers to "chain" together different components—such as prompt templates, memory modules, and document loaders—to create sophisticated workflows.
By using LangChain, a